# Parametric sensitivity analysis

Experiments A–C hold input voltage at 230 V RMS, frequency at 50 Hz and delivered apparent power at 100 VA. They reuse Stage 2 geometry, circuit, material equations and feasibility screens. The solved 100 VA high-voltage branch keeps power fixed even when terminal voltage differs from 24 V. All losses use the calculated loaded peak flux density. The 241.5 V flux screen is a separate no-drop check. Experiment D varies input voltage at fixed delivered power and unchanged turns, allowing output voltage to move.

Read the [Stage 2 model](../report/04_model_development.md) and [Stage 3 report](../report/05_sensitivity_analysis.md) for assumptions and figure captions. Run the notebook in order to regenerate the CSV files and figures.

In [1]:
from collections import Counter
from csv import DictReader, DictWriter
from dataclasses import replace
from math import ceil, floor, isclose, nan
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from src.constraints import DesignLimits, check_design
from src.core_geometry import EICore, WindowClearances
from src.material_models import Copper, HysteresisScenario, LaminatedSteel
from src.transformer_model import (
    ElectricalSpec, Winding, evaluate_design, ideal_secondary_voltage,
    nearest_secondary_turns, peak_flux_density, resistive_load_point,
)

data = root / "data" / "sensitivity_results"
figures = root / "figures" / "sensitivity"
data.mkdir(parents=True, exist_ok=True)
figures.mkdir(parents=True, exist_ok=True)

with (root / "data" / "baseline_parameters.csv").open(encoding="utf-8", newline="") as handle:
    p = {row["parameter"]: float(row["value"]) for row in DictReader(handle)}

spec = ElectricalSpec(p["primary_rms_v"], p["secondary_target_rms_v"],
                      p["frequency_hz"], p["rated_va"])
assert p["load_power_factor"] == 1
depth = p["effective_area_m2"] / (p["stacking_factor"] * p["centre_width_m"])
base_core = EICore(p["centre_width_m"], depth, p["window_width_m"], p["window_height_m"],
                   p["stacking_factor"])
clearances = WindowClearances(p["former_wall_m"], p["outer_clearance_m"],
                              p["end_margin_m"], p["winding_barrier_m"])
base_primary, base_secondary = (
    Winding(int(p[f"{name}_turns"]), p[f"{name}_bare_diameter_m"], p[f"{name}_insulated_diameter_m"],
            p[f"{name}_lead_length_m"], p["winding_temperature_c"])
    for name in ("primary", "secondary")
)
copper = Copper(p["copper_resistivity_20_ohm_m"], p["copper_temp_coefficient_per_k"])
steel = LaminatedSteel(p["sheet_thickness_m"], p["steel_resistivity_ohm_m"],
                       p["steel_density_kg_m3"])
limits = DesignLimits(p["flux_limit_t"], p["maximum_supply_rms_v"], p["minimum_frequency_hz"],
                      p["maximum_fill_fraction"], p["ideal_voltage_tolerance_fraction"],
                      p["loaded_voltage_tolerance_fraction"], p["maximum_current_density_a_m2"])
anchor_f = p["anchor_frequency_hz"]
anchor_b = p["anchor_high_polarisation_t"]
anchor_remainder = p["anchor_high_total_loss_w_kg"] - steel.classical_loss_w_kg(anchor_f, anchor_b)
assert anchor_remainder > 0
scenarios = {
    level: HysteresisScenario(
        f"Stage 2 illustrative {level}", p[f"hysteresis_{level}_fraction"] * anchor_remainder / anchor_f,
        anchor_b, p["hysteresis_exponent"], p["anchor_low_polarisation_t"], anchor_b,
    )
    for level in ("low", "mid", "high")
}
base = evaluate_design(spec, base_core, clearances, base_primary, base_secondary, copper, steel,
                       scenarios["mid"], fixed_va=True)


def write_csv(name, rows):
    path = data / f"{name}.csv"
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = DictWriter(handle, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


def read_csv(name):
    with (data / f"{name}.csv").open(encoding="utf-8", newline="") as handle:
        return list(DictReader(handle))


def scenario_columns(frequency, peak, mass, copper_w, classical_w):
    covered = scenarios["mid"].contains(peak)
    row = {"hysteresis_covered": covered}
    for level, scenario in scenarios.items():
        hysteresis_w = mass * scenario.specific_loss_w_kg(frequency, peak) if covered else None
        row[f"hysteresis_{level}_w"] = hysteresis_w
        row[f"total_{level}_w"] = copper_w + classical_w + hysteresis_w if covered else None
    return row


def geometry_row(supply, core, primary, secondary, geometry, *, screen_main=True):
    result = evaluate_design(supply, core, clearances, primary, secondary, copper, steel,
                             scenarios["mid"], fixed_va=True)
    load = result.load
    j_primary = load.primary_rms_a / primary.conductor_area_m2
    j_secondary = load.secondary_rms_a / secondary.conductor_area_m2
    row = dict(
        geometry=geometry, primary_rms_v=supply.primary_rms_v, frequency_hz=supply.frequency_hz,
        rated_va=supply.rated_va, primary_turns=primary.turns, secondary_turns=secondary.turns,
        effective_area_cm2=round(core.effective_area_m2 * 1e4, 12),
        centre_width_mm=core.centre_width_m * 1e3, stack_depth_mm=core.stack_depth_m * 1e3,
        window_width_mm=core.window_width_m * 1e3, window_height_mm=core.window_height_m * 1e3,
        path_length_mm=core.magnetic_path_m * 1e3, steel_volume_cm3=core.steel_volume_m3 * 1e6,
        core_mass_kg=result.core_mass_kg, primary_mlt_mm=result.primary_layout.mean_turn_length_m * 1e3,
        secondary_mlt_mm=result.secondary_layout.mean_turn_length_m * 1e3,
        primary_layers=result.primary_layout.layers, secondary_layers=result.secondary_layout.layers,
        primary_ohm=result.primary_ohm, secondary_ohm=result.secondary_ohm,
        primary_current_a=load.primary_rms_a, secondary_current_a=load.secondary_rms_a,
        primary_current_density_a_mm2=j_primary / 1e6, secondary_current_density_a_mm2=j_secondary / 1e6,
        ideal_secondary_v=result.ideal_secondary_rms_v, terminal_secondary_v=load.secondary_terminal_rms_v,
        ideal_voltage_error_fraction=result.ideal_secondary_rms_v / supply.secondary_target_rms_v - 1,
        loaded_voltage_error_fraction=load.secondary_terminal_rms_v / supply.secondary_target_rms_v - 1,
        delivered_va=load.delivered_va, no_drop_peak_t=result.no_drop_peak_t, loaded_peak_t=result.loaded_peak_t,
        usable_window_mm2=result.usable_window_m2 * 1e6, usable_window_width_mm=result.usable_width_m * 1e3,
        radial_build_mm=result.required_radial_build_m * 1e3, bare_copper_fill_fraction=result.bare_copper_fill,
        primary_copper_w=result.primary_copper_w, secondary_copper_w=result.secondary_copper_w,
        copper_w=result.copper_w, classical_w=result.classical_w,
    )
    row.update(scenario_columns(supply.frequency_hz, result.loaded_peak_t, result.core_mass_kg,
                                result.copper_w, result.classical_w))
    if row["hysteresis_covered"]:
        assert isclose(row["total_mid_w"], result.separated_total_w)
    assert isclose(load.delivered_va, supply.rated_va)
    if screen_main:
        checks = check_design(result, limits)
        failures = [check.name for check in checks if not check.passed]
        row.update(screening_peak_t=checks[0].value, design_feasible=not failures,
                   constraint_failures=" | ".join(failures),
                   feasible_with_complete_losses=not failures and row["hysteresis_covered"])
    else:
        row.update(
            current_density_screen_pass=max(j_primary, j_secondary) <= limits.maximum_current_density_a_m2,
            nominal_no_drop_flux_screen_pass=result.no_drop_peak_t <= limits.peak_flux_density_t,
        )
    return row

## A and B · Controlled geometry-aware sweeps

A uses 480–900 primary turns in increments of five at 12 cm². At each primary turn count, enumerate secondary integers inside the ±1% ideal-ratio window and select the one that minimises absolute loaded voltage error. The tie-break is smaller absolute ideal error and then fewer secondary turns. All evaluated options are saved with their selection status.

B uses 9–18 cm² in increments of 0.125 cm² at the fixed 600:63 winding pair. Compare proportional dimension scaling with a fixed stamping that varies stack depth alone. Fixed bobbin deductions, wire sizes, winding temperature, lead allowances and material apply throughout. The layout and mass are recalculated for each point. A numerical result can fail design screens, and its hysteresis value is available only between 1.0 and 1.5 T loaded induction.

In [2]:
a_rows, a_options = [], []
for np in range(480, 901, 5):
    primary = replace(base_primary, turns=np)
    centre = np * spec.secondary_target_rms_v / spec.primary_rms_v
    tolerance = limits.ideal_voltage_tolerance_fraction
    allowed = range(max(1, floor(centre * (1 - tolerance))),
                    ceil(centre * (1 + tolerance)) + 1)
    valid_turns = [
        ns for ns in allowed
        if abs(ideal_secondary_voltage(spec.primary_rms_v, np, ns)
               / spec.secondary_target_rms_v - 1) <= tolerance
    ]
    fallback = not valid_turns
    if fallback:
        valid_turns = [nearest_secondary_turns(spec.primary_rms_v, spec.secondary_target_rms_v, np)]
    options = [
        geometry_row(spec, base_core, primary, replace(base_secondary, turns=ns), "fixed core")
        for ns in valid_turns
    ]
    selected = min(options, key=lambda row: (abs(row["loaded_voltage_error_fraction"]),
                                              abs(row["ideal_voltage_error_fraction"]),
                                              row["secondary_turns"]))
    for row in options:
        row.update(ideal_ratio_fallback=fallback, selected=row is selected)
    a_rows.append(selected)
    a_options.extend(options)

b_rows = []
for index in range(73):
    area_m2 = (9 + index * 0.125) * 1e-4
    for label, core in (
        ("proportional dimensions", base_core.with_effective_area(area_m2)),
        ("fixed stamping", replace(base_core, stack_depth_m=area_m2 /
                                    (base_core.stacking_factor * base_core.centre_width_m))),
    ):
        b_rows.append(geometry_row(spec, core, base_primary, base_secondary, label))

write_csv("a_winding_turns", a_rows)
write_csv("a_secondary_turn_candidates", a_options)
write_csv("b_effective_core_area", b_rows)
assert len(a_rows) == 85 and len(b_rows) == 146
assert sum(row["selected"] for row in a_options) == len(a_rows)
for label, rows in (
    ("A", a_rows),
    ("B proportional", [row for row in b_rows if row["geometry"] == "proportional dimensions"]),
    ("B fixed stamping", [row for row in b_rows if row["geometry"] == "fixed stamping"]),
):
    feasible = [row for row in rows if row["design_feasible"]]
    complete = [row for row in rows if row["feasible_with_complete_losses"]]
    failures = Counter(reason for row in rows for reason in row["constraint_failures"].split(" | ") if reason)
    print(f"{label}: {len(rows)} points, {len(feasible)} screen passes, "
          f"{len(complete)} passes with all losses. Screen failures: {dict(failures)}")
    if complete:
        print("  Complete pass coordinates:", [
            row["primary_turns"] if label == "A" else round(row["effective_area_cm2"], 4)
            for row in complete
        ])

A: 85 points, 0 screen passes, 0 passes with all losses. Screen failures: {'Peak flux at screening envelope': 25, 'Loaded secondary voltage error': 82, 'Secondary current density': 38, 'Bare-copper window fill': 3}
B proportional: 73 points, 0 screen passes, 0 passes with all losses. Screen failures: {'Peak flux at screening envelope': 25, 'Loaded secondary voltage error': 73}
B fixed stamping: 73 points, 0 screen passes, 0 passes with all losses. Screen failures: {'Peak flux at screening envelope': 25, 'Loaded secondary voltage error': 73}


## C · Frozen mass and mean turn length

For each A point and each proportional-geometry B point, freeze the steel mass and both mean turn lengths at the common Stage 2 baseline. Keep the same integer turns, effective area and electrical requirements as the corresponding geometry-aware case. Recalculate resistance using Stage 2 winding methods, solve the same resistive circuit and compute losses from the resulting loaded induction. The frozen case is a counterfactual without a consistent window model. The linked feasibility flag refers to its geometry-aware counterpart only.

In [3]:
comparison = []
for sweep, rows in (
    ("A", a_rows),
    ("B", [row for row in b_rows if row["geometry"] == "proportional dimensions"]),
):
    for aware in rows:
        primary = replace(base_primary, turns=aware["primary_turns"])
        secondary = replace(base_secondary, turns=aware["secondary_turns"])
        p_length = base.primary_layout.mean_turn_length_m
        s_length = base.secondary_layout.mean_turn_length_m
        p_ohm = primary.resistance_ohm(copper, p_length)
        s_ohm = secondary.resistance_ohm(copper, s_length)
        load = resistive_load_point(spec, secondary.turns / primary.turns, p_ohm, s_ohm, fixed_va=True)
        peak = peak_flux_density(load.primary_induced_rms_v, spec.frequency_hz, primary.turns,
                                 aware["effective_area_cm2"] * 1e-4)
        p_cu = load.primary_rms_a**2 * p_ohm
        s_cu = load.secondary_rms_a**2 * s_ohm
        eddy = base.core_mass_kg * steel.classical_loss_w_kg(spec.frequency_hz, peak)
        row = dict(
            sweep=sweep, primary_rms_v=spec.primary_rms_v, frequency_hz=spec.frequency_hz, rated_va=spec.rated_va,
            primary_turns=primary.turns, secondary_turns=secondary.turns,
            effective_area_cm2=aware["effective_area_cm2"], core_mass_kg=base.core_mass_kg,
            primary_mlt_mm=p_length * 1e3, secondary_mlt_mm=s_length * 1e3,
            primary_ohm=p_ohm, secondary_ohm=s_ohm, primary_current_a=load.primary_rms_a,
            secondary_current_a=load.secondary_rms_a, terminal_secondary_v=load.secondary_terminal_rms_v,
            delivered_va=load.delivered_va, loaded_peak_t=peak, primary_copper_w=p_cu, secondary_copper_w=s_cu,
            copper_w=p_cu + s_cu, classical_w=eddy, geometry_aware_counterpart_feasible=aware["design_feasible"],
        )
        row.update(scenario_columns(spec.frequency_hz, peak, base.core_mass_kg, p_cu + s_cu, eddy))
        if isclose(row["effective_area_cm2"], p["effective_area_m2"] * 1e4) and primary.turns == base_primary.turns:
            assert isclose(row["total_mid_w"], base.separated_total_w, rel_tol=1e-12)
            assert isclose(row["total_mid_w"], aware["total_mid_w"], rel_tol=1e-12)
        comparison.append(row)

write_csv("c_frozen_geometry_comparison", comparison)
print(f"C: {len(comparison)} matched counterfactual points. Both sweeps agree at the baseline.")

C: 158 matched counterfactual points. Both sweeps agree at the baseline.


## D · Optional fixed-build voltage experiment

Vary supply over 190–250 V RMS in increments of 5 V at 50 Hz, keeping the same 600:63 build and exact 100 VA output. The turns ratio stays fixed and secondary terminal voltage is allowed to change. The 24 V target and fixed-input screening envelope from the main study do not constrain this hypothetical voltage experiment. Nominal no-drop flux and conductor current density are flagged against the same provisional thresholds at each supply. This experiment contributes no candidates to A–C.

In [4]:
voltage_rows = [
    geometry_row(replace(spec, primary_rms_v=voltage), base_core, base_primary, base_secondary,
                 "fixed E–I build", screen_main=False)
    for voltage in range(190, 251, 5)
]
write_csv("d_input_voltage_fixed_va", voltage_rows)
for row in voltage_rows[::4]:
    print(f'{row["primary_rms_v"]:.0f} V in | {row["terminal_secondary_v"]:.3f} V out | '
          f'{row["loaded_peak_t"]:.3f} T | {row["copper_w"]:.3f} W copper')

190 V in | 17.956 V out | 1.143 T | 11.102 W copper
210 V in | 20.285 V out | 1.273 T | 8.700 W copper
230 V in | 22.563 V out | 1.402 T | 7.031 W copper
250 V in | 24.807 V out | 1.530 T | 5.817 W copper


## Figures and CSV verification

Figures read the exported files first. Hollow markers on the main loss plots identify cases that fail one or more numerical design screens. Missing hysteresis and total losses appear as breaks in lines. Coloured ranges on total-loss plots span the Stage 2 low and high scenarios. They are not uncertainty bounds. Each figure is written as a vector SVG and a 360 dpi PNG.

In [5]:
colors = {
    "copper": "#A84A12", "classical": "#00727B", "hysteresis": "#72378D",
    "total": "#173754", "muted": "#566472", "limit": "#9D3442",
}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10.5, "axes.titlesize": 12,
    "axes.labelsize": 11, "axes.spines.top": False, "axes.spines.right": False,
    "axes.axisbelow": True, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": "#D9E1E8", "grid.linewidth": 0.65, "lines.linewidth": 2,
    "legend.frameon": False, "svg.fonttype": "none",
})
a = read_csv("a_winding_turns")
b = read_csv("b_effective_core_area")
c = read_csv("c_frozen_geometry_comparison")
d = read_csv("d_input_voltage_fixed_va")
plot_checks = []


def values(rows, field):
    return [float(row[field]) if row[field] else nan for row in rows]


def series(ax, rows, x_field, y_field, label, color, **style):
    x = values(rows, x_field)
    y = values(rows, y_field)
    line, = ax.plot(x, y, label=label, color=color, **style)
    plot_checks.append((line, rows, x_field, y_field))
    return line


def mark_failures(ax, rows, x_field, y_field, color, label=None):
    rejected = [row for row in rows if row["design_feasible"] == "False" and row[y_field]]
    if rejected:
        ax.scatter(values(rejected, x_field), values(rejected, y_field), s=22,
                   facecolors="white", edgecolors=color, linewidths=0.9, zorder=4, label=label)


def shade_scenarios(ax, rows, x_field):
    ax.fill_between(values(rows, x_field), values(rows, "total_low_w"), values(rows, "total_high_w"),
                    color=colors["total"], alpha=0.13, label="Low–high scenarios")


def save(fig, filename):
    for extension in ("svg", "png"):
        fig.savefig(figures / f"{filename}.{extension}", dpi=360, facecolor="white")
    plt.close(fig)


fig, ax = plt.subplots(figsize=(7.4, 4.35), layout="constrained")
series(ax, a, "primary_turns", "loaded_peak_t", "Loaded at 100 VA", colors["total"])
series(ax, a, "primary_turns", "no_drop_peak_t", "Nominal no-drop", colors["classical"], linestyle="--")
series(ax, a, "primary_turns", "screening_peak_t", "High-supply no-drop", colors["muted"], linestyle=":")
ax.axhline(limits.peak_flux_density_t, color=colors["limit"], linewidth=1.4, label="Provisional screen")
ax.set(xlabel="Primary turns", ylabel="Peak flux density (T)")
ax.legend(ncols=2, loc="upper right")
save(fig, "a_peak_flux_density_vs_primary_turns")

fig, ax = plt.subplots(figsize=(7.4, 4.35), layout="constrained")
for field, label, color in (
    ("copper_w", "Winding copper", colors["copper"]),
    ("classical_w", "Classical eddy current", colors["classical"]),
    ("hysteresis_mid_w", "Illustrative hysteresis (mid)", colors["hysteresis"]),
):
    series(ax, a, "primary_turns", field, label, color)
    mark_failures(ax, a, "primary_turns", field, color)
ax.scatter([], [], s=22, facecolors="white", edgecolors=colors["muted"], label="Fails ≥1 screen")
ax.set(xlabel="Primary turns", ylabel="Loss (W)")
ax.legend(loc="upper left")
save(fig, "a_component_losses_vs_primary_turns")

fig, ax = plt.subplots(figsize=(7.4, 4.35), layout="constrained")
shade_scenarios(ax, a, "primary_turns")
series(ax, a, "primary_turns", "total_mid_w", "Middle scenario total", colors["total"])
mark_failures(ax, a, "primary_turns", "total_mid_w", colors["total"], "Fails ≥1 screen")
passes = [row for row in a if row["feasible_with_complete_losses"] == "True"]
if passes:
    ax.scatter(values(passes, "primary_turns"), values(passes, "total_mid_w"), s=34,
               color=colors["classical"], zorder=5, label="Passes all screens")
ax.set(xlabel="Primary turns", ylabel="Three-loss sum (W)", xlim=(465, 915))
ax.legend(loc="upper left")
save(fig, "a_total_modelled_loss_vs_primary_turns")

b_proportional = [row for row in b if row["geometry"] == "proportional dimensions"]
b_stamping = [row for row in b if row["geometry"] == "fixed stamping"]
fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.45), sharey=True, layout="constrained")
for ax, rows, title in zip(axes, (b_proportional, b_stamping), ("Proportional dimensions", "Fixed stamping")):
    for field, label, color in (
        ("copper_w", "Winding copper", colors["copper"]),
        ("classical_w", "Classical eddy current", colors["classical"]),
        ("hysteresis_mid_w", "Illustrative hysteresis (mid)", colors["hysteresis"]),
        ("total_mid_w", "Middle scenario total", colors["total"]),
    ):
        series(ax, rows, "effective_area_cm2", field, label, color)
    mark_failures(ax, rows, "effective_area_cm2", "total_mid_w", colors["total"], "Total fails ≥1 screen")
    ax.set(xlabel="Effective core area (cm²)", title=title)
axes[0].set_ylabel("Loss (W)")
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncols=3, fontsize=9)
save(fig, "b_losses_vs_effective_core_area")

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.15), sharey=True, layout="constrained")
for ax, sweep, rows, x_field, title, xlabel in (
    (axes[0], "A", a, "primary_turns", "Vary winding turns", "Primary turns"),
    (axes[1], "B", b_proportional, "effective_area_cm2", "Vary effective core area", "Effective core area (cm²)"),
):
    frozen = [row for row in c if row["sweep"] == sweep]
    series(ax, rows, x_field, "total_mid_w", "Geometry-aware", colors["total"])
    series(ax, frozen, x_field, "total_mid_w", "Frozen mass and turn lengths",
           colors["copper"], linestyle="--")
    mark_failures(ax, rows, x_field, "total_mid_w", colors["total"], "Geometry-aware fails ≥1 screen")
    ax.set(xlabel=xlabel, title=title, xlim=(465, 915) if sweep == "A" else (8.75, 18.25))
axes[0].set_ylabel("Middle scenario total loss (W)")
axes[0].legend(loc="upper left", fontsize=8.5)
save(fig, "c_frozen_vs_geometry_aware_total_loss")

fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.55), layout="constrained")
for ax, field, ylabel, color in (
    (axes[0], "terminal_secondary_v", "Output voltage (V RMS)", colors["total"]),
    (axes[1], "loaded_peak_t", "Loaded peak flux density (T)", colors["classical"]),
    (axes[2], "copper_w", "Winding copper loss (W)", colors["copper"]),
):
    series(ax, d, "primary_rms_v", field, ylabel, color)
    ax.set(xlabel="Input voltage (V RMS)", ylabel=ylabel)
save(fig, "d_fixed_va_input_voltage_sensitivity")

for line, rows, x, y in plot_checks:
    for key, actual in ((x, line.get_xdata()), (y, line.get_ydata())):
        expected = values(rows, key)
        assert len(actual) == len(expected)
        assert all((a == e) or (a != a and e != e) for a, e in zip(actual, expected))
for name, rows, count in (("A", a, 85), ("B", b, 146), ("C", c, 158), ("D", d, 13)):
    assert len(rows) == count
    assert all(isclose(float(row["delivered_va"]), 100) for row in rows)
    for row in rows:
        for level in scenarios:
            total = row[f"total_{level}_w"]
            assert bool(total) == (row["hysteresis_covered"] == "True")
            if total:
                assert isclose(float(total), sum(float(row[field]) for field in
                                                 ("copper_w", "classical_w", f"hysteresis_{level}_w")))
print(f"Verified {len(plot_checks)} plotted lines against the saved CSV values.")
print("Saved six figures as SVG and 360 dpi PNG.")

Verified 22 plotted lines against the saved CSV values.
Saved six figures as SVG and 360 dpi PNG.
